## Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

- Role – Identifies the message type (e.g. system, user)
- Content – Represents the actual content of the message (like text, images, audio, documents, etc.)
- Metadata – Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [11]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")

### Text Prompts

Text prompts are strings - ideal for straightforward generation tasks where you don't need to retain conversation history.

In [12]:
model.invoke("What is langchain")

AIMessage(content='<think>\nOkay, the user is asking what LangChain is. Let me start by recalling my knowledge. LangChain is a framework for developing applications with large language models (LLMs). It provides tools for integrating LLMs with other components like databases, APIs, and more. But I need to explain this in a clear and structured way.\n\nFirst, I should mention the core purpose: building applications powered by LLMs. Then, outline the main components. The user might be a developer or someone looking to use LLMs in their projects, so technical details are important but should be explained simply.\n\nLangChain has modules like Prompt, LLM, Chains, Agents, Memory, Callbacks, and more. Each of these components serves a specific function. For example, Chains allow combining multiple LLM calls or integrating with other data sources. Agents can handle more complex tasks by interacting with environments.\n\nUse cases are also important. Maybe the user wants to know how LangChain 

Use text prompts when:
- You have a single, standalone request
- You don't need conversation history
- You want minimal code complexity

### Message Prompts

Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types
- System message – Tells the model how to behave and provide context for interactions.
- Human message – Represents user input and interactions with the model.
- AI message – Responses generated by the model, including text content, tool calls, and metadata.
- Tool message – Represents the outputs of tool calls.

In [13]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on Love")
]

response=model.invoke(messages)
response.content

'<think>\nOkay, the user wants a poem about love. Let me start by brainstorming some common themes associated with love—passion, connection, unity, maybe even challenges. But I should avoid clichés. Maybe use nature metaphors? Like the sun, ocean, or trees to symbolize different aspects of love.\n\nFirst stanza: Maybe start with something timeless, like the sun or stars. "In the quiet glow of twilight\'s embrace" sets a peaceful scene. Introduce love as a whisper or a spark. Use imagery that conveys subtlety and warmth.\n\nSecond stanza: Contrast with something more dynamic. Maybe the ocean to represent depth and movement. "Love is the tide that bends the shore" shows persistence and adaptability. Emphasize how love shapes us over time.\n\nThird stanza: Introduce the idea of connection between two people. Use celestial imagery, like constellations. "Two souls entwined in cosmic dance" suggests harmony and fate. Mention laughter and tears to show the full range of emotions.\n\nFourth st

In [14]:
system_msg = SystemMessage("You are a helpful coding assistant.")

messages=[
    system_msg,
    HumanMessage("How do I create a REST API?")
]

response=model.invoke(messages)
response.content

'<think>\nOkay, the user is asking how to create a REST API. Let me break down what they need to know. First, I should explain what a REST API is, in case they\'re not familiar. Then, outline the steps involved in creating one. But wait, they might not have a specific framework in mind. Maybe I should mention different languages and frameworks. Wait, but the answer should be general enough, maybe with examples in a common framework. Let\'s pick one that\'s widely used, like Python with Flask. That\'s a good starting point for beginners.\n\nSo the steps would be: setting up the environment, defining endpoints, handling HTTP methods (GET, POST, PUT, DELETE), working with JSON data, error handling, and maybe adding some middleware for production. Oh, they might also need to know how to test the API. Tools like Postman or curl could be useful. Also, documentation is important. Should I mention using something like Swagger?\n\nWait, the user might not know about the structure of a RESTful A

In [ ]:
## Detailed info to the LLM through System message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]

response = model.invoke(messages)
print(response.content)

In [16]:
## Message Metadata
human_msg = HumanMessage(
    content="Hello!",
    name="Umang",
    id="msg_123",
)

In [17]:
response = model.invoke([
    human_msg
])
response

AIMessage(content='<think>\nOkay, the user greeted me with Hello. I need to respond politely. Let me check the guidelines. I should be friendly and offer help. Maybe say Hello back and ask how I can assist them. Keep it simple and open-ended.\n</think>\n\nHello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 10, 'total_tokens': 74, 'completion_time': 0.124841218, 'completion_tokens_details': None, 'prompt_time': 0.00050093, 'prompt_tokens_details': None, 'queue_time': 0.16081874, 'total_time': 0.125342148}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f0ec0-63b0-7e40-900f-84948f23ee16-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 64, 'total_tokens': 74})

In [ ]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

In [ ]:
response.usage_metadata

In [20]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)

ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"

tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,      # Model's tool call
    tool_message,    # Tool execution result
]

response = model.invoke(messages)  # Model processes the result

In [21]:
tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_123')

In [22]:
response

AIMessage(content='<think>\nOkay, the user asked for the weather in San Francisco. I called the get_weather function with the location set to San Francisco. The response came back as "Sunny, 72°F". Now I need to present this information in a friendly and clear way.\n\nFirst, I should confirm the location to make sure there\'s no confusion. Then, state the current weather condition and temperature. It\'s good to add a note about what the temperature feels like, maybe mention if it\'s a good day for outdoor activities. Keep it concise but helpful. Let me check if there are any additional details I should include, like wind speed or humidity, but the response only provided sunny and 72°F. So I\'ll stick to that. Make sure the tone is positive since the weather is sunny. Alright, let\'s put it all together.\n</think>\n\nThe current weather in San Francisco is **sunny** with a temperature of **72°F**. It\'s a perfect day for outdoor activities! ☀️', additional_kwargs={}, response_metadata={